In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
%matplotlib inline

In [ ]:
customers = pd.read_csv('maintenance_cleaned.csv')

In [ ]:
display(customers.head())

,machine_type,vibration_rms,temperature_motor,current_phase_avg,pressure_level,rpm,operating_mode,hours_since_maintenance,ambient_temp,failure_within_24h
0,CNC,0.81,49.51,5.10,23.6,860.9,idle,273.80,13.9,0
1,CNC,0.75,40.58,5.30,23.6,899.6,idle,273.85,10.2,0
2,CNC,0.71,49.70,6.43,21.3,862.7,idle,274.15,13.6,0
3,CNC,0.76,43.04,4.79,22.6,870.4,idle,274.55,13.4,0
4,CNC,0.88,41.39,4.44,22.2,881.9,idle,274.70,10.8,0


In [ ]:
display(customers.describe())

,vibration_rms,temperature_motor,current_phase_avg,pressure_level,rpm,hours_since_maintenance,ambient_temp,failure_within_24h
count,24042.000000,24042.000000,24042.000000,24042.000000,24042.000000,24042.000000,24042.000000,24042.000000
mean,1.605370,51.357662,8.751044,58.523667,1138.445662,172.630624,12.996398,0.148074
std,1.041905,12.302671,5.300137,38.050389,903.498629,150.722469,2.883994,0.355181
min,0.350000,28.000000,2.200000,10.100000,124.100000,0.000000,8.000000,0.000000
25%,0.850000,42.890000,4.710000,23.100000,496.025000,42.870000,10.500000,0.000000
50%,1.270000,50.060000,6.430000,46.300000,856.000000,121.610000,13.000000,0.000000
75%,2.230000,59.600000,12.990000,90.800000,1667.875000,295.575000,15.500000,0.000000
max,6.370000,95.000000,35.000000,206.500000,4098.800000,575.630000,18.000000,1.000000


In [ ]:
display(customers.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24042 entries, 0 to 24041
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   machine_type             24042 non-null  object 
 1   vibration_rms            24042 non-null  float64
 2   temperature_motor        24042 non-null  float64
 3   current_phase_avg        24042 non-null  float64
 4   pressure_level           24042 non-null  float64
 5   rpm                      24042 non-null  float64
 6   operating_mode           24042 non-null  object 
 7   hours_since_maintenance  24042 non-null  float64
 8   ambient_temp             24042 non-null  float64
 9   failure_within_24h       24042 non-null  int64  
dtypes: float64(7), int64(1), object(2)
memory usage: 1.8+ MB


None

In [ ]:
from sklearn.model_selection import train_test_split

X = customers[[
    "machine_type",
    "vibration_rms",
    "temperature_motor",
    "current_phase_avg",
    "pressure_level",
    "rpm",
    "operating_mode",
    "hours_since_maintenance",
    "ambient_temp"
]]
y = customers["failure_within_24h"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=101, stratify=y)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ["vibration_rms", "temperature_motor", "current_phase_avg", "pressure_level", "rpm", "hours_since_maintenance", "ambient_temp"]
categorical_features = ["machine_type", "operating_mode"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [ ]:
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=101))
])

In [ ]:
print("Entraînement en cours...")
pipeline_rf.fit(X_train, y_train)
print("Modèle entraîné avec succès !")

Entraînement en cours...
Modèle entraîné avec succès !


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = pipeline_rf.predict(X_test)

print("\n--- Rapport de Performance ---")
print(classification_report(y_test, y_pred))


--- Rapport de Performance ---
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      6145
           1       0.89      0.86      0.87      1068

    accuracy                           0.96      7213
   macro avg       0.93      0.92      0.93      7213
weighted avg       0.96      0.96      0.96      7213



In [ ]:
joblib.dump(pipeline_rf, 'random_forest_model.pkl')
print("\nPipeline sauvegardé dans 'random_forest_model.pkl'")


Pipeline sauvegardé dans 'random_forest_model.pkl'
